# Chapter 7
Use the `tfq` environment described in the repository README. Restart the kernel and run all cells in order. The code cells reproduce the final book listings.


In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import cirq
import numpy as np
import sympy
import tensorflow as tf
import tensorflow_quantum as tfq
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

tf.keras.utils.set_random_seed(7)
X, y = make_classification(
    n_samples=80, n_features=2, n_informative=2,
    n_redundant=0, n_clusters_per_class=1, random_state=7,
)
x_train, x_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=7,
)
scaler = StandardScaler().fit(x_train)
x_train_scaled = np.clip(
    scaler.transform(x_train), -np.pi, np.pi
).astype("float32")
x_valid_scaled = np.clip(
    scaler.transform(x_valid), -np.pi, np.pi
).astype("float32")
y_train = y_train.astype("float32").reshape(-1, 1)
y_valid = y_valid.astype("float32").reshape(-1, 1)

def dense_layer(units, activation=None, seed=13):
    """Create a dense layer with an explicit integer initializer seed."""
    return tf.keras.layers.Dense(
        units, activation=activation,
        kernel_initializer=tf.keras.initializers.GlorotUniform(
            seed=seed
        ),
    )

qubits = cirq.GridQubit.rect(1, 2)
theta = sympy.symbols("theta0:4")

def encode_example(x):
    """Map a pair of input angles to a fixed data circuit."""
    return cirq.Circuit(
        cirq.ry(float(x[0]))(qubits[0]),
        cirq.ry(float(x[1]))(qubits[1]),
    )

def trainable_circuit():
    """Return the two-qubit ansatz with four trainable angles."""
    return cirq.Circuit(
        cirq.ry(theta[0])(qubits[0]),
        cirq.ry(theta[1])(qubits[1]),
        cirq.CZ(*qubits),
        cirq.rx(theta[2])(qubits[0]),
        cirq.rx(theta[3])(qubits[1]),
    )

observables = [cirq.Z(q) for q in qubits]
quantum_inputs = tf.keras.Input(shape=(), dtype=tf.string)
quantum_layer = tfq.layers.PQC(
    trainable_circuit(), observables,
    backend="noiseless", repetitions=None,
    differentiator=tfq.differentiators.ParameterShift(),
    initializer=tf.keras.initializers.RandomUniform(seed=7),
    name="quantum_features",
)
quantum_features = quantum_layer(quantum_inputs)
outputs = dense_layer(1, activation="sigmoid")(
    quantum_features
)
model = tf.keras.Model(quantum_inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(0.02),
    loss="binary_crossentropy", metrics=["accuracy"],
)
train_circuits = tfq.convert_to_tensor(
    [encode_example(x) for x in x_train_scaled]
)
valid_circuits = tfq.convert_to_tensor(
    [encode_example(x) for x in x_valid_scaled]
)
history = model.fit(
    train_circuits, y_train, batch_size=16, epochs=20, verbose=0,
    validation_data=(valid_circuits, y_valid),
)
valid_probabilities = model.predict(valid_circuits, verbose=0)
valid_labels = (valid_probabilities >= 0.5).astype("int32")
print("validation accuracy:", accuracy_score(y_valid, valid_labels))


In [ ]:
shift_layer = tfq.layers.PQC(
    trainable_circuit(), observables,
    backend="noiseless", repetitions=None,
    differentiator=tfq.differentiators.ParameterShift(),
    initializer=tf.keras.initializers.Constant(0.1),
    name="parameter_shift_probe",
)
with tf.GradientTape() as tape:
    probe_outputs = shift_layer(train_circuits[:2])
    probe_loss = tf.reduce_sum(probe_outputs)
probe_gradients = tape.gradient(probe_loss, shift_layer.trainable_variables)
assert all(g is not None for g in probe_gradients)
print("output shape:", probe_outputs.shape)
print("gradient norm:", float(tf.linalg.global_norm(probe_gradients)))

experiment_card = {
    "quantum_layer": "tfq.layers.PQC",
    "differentiator": "ParameterShift",
    "num_qubits": len(qubits),
    "num_symbols": len(theta),
    "num_observables": len(observables),
    "backend": "noiseless", "repetitions": None,
}


In [ ]:
tf.keras.utils.set_random_seed(7)
input_dim = x_train_scaled.shape[1]
num_controls = len(theta)
feature_inputs = tf.keras.Input(shape=(input_dim,), name="features")
circuit_inputs = tf.keras.Input(shape=(), dtype=tf.string, name="circuits")
controller = tf.keras.Sequential([
    dense_layer(16, activation="relu", seed=17),
    dense_layer(num_controls, seed=19),
], name="classical_controller")
symbol_values = controller(feature_inputs)
controlled_layer = tfq.layers.ControlledPQC(
    trainable_circuit(), observables,
    backend="noiseless", repetitions=None,
    differentiator=tfq.differentiators.ParameterShift(),
    name="controlled_quantum_layer",
)
controlled_expectations = controlled_layer(
    [circuit_inputs, symbol_values]
)
outputs = dense_layer(1, activation="sigmoid")(
    controlled_expectations
)
hybrid_model = tf.keras.Model(
    inputs=[feature_inputs, circuit_inputs], outputs=outputs,
)
hybrid_model.compile(
    optimizer=tf.keras.optimizers.Adam(0.02),
    loss="binary_crossentropy", metrics=["accuracy"],
)
hybrid_history = hybrid_model.fit(
    [x_train_scaled, train_circuits], y_train,
    batch_size=16, epochs=20, verbose=0,
    validation_data=([x_valid_scaled, valid_circuits], y_valid),
)
hybrid_probabilities = hybrid_model.predict(
    [x_valid_scaled, valid_circuits], verbose=0,
)
print("controlled validation accuracy:", accuracy_score(
    y_valid, hybrid_probabilities >= 0.5
))


In [ ]:
import time

def build_ablation(kind):
    """Build one branch variant; return its model and quantum-input flag."""
    tf.keras.utils.set_random_seed(7)
    features = tf.keras.Input(shape=(2,))
    shared = dense_layer(4, activation="tanh", seed=7)(features)
    inputs = features
    uses_quantum = kind in ("fixed_quantum", "trainable_quantum")
    if uses_quantum:
        circuits = tf.keras.Input(shape=(), dtype=tf.string)
        branch = tfq.layers.PQC(
            trainable_circuit(), observables,
            backend="noiseless", repetitions=None,
            differentiator=tfq.differentiators.ParameterShift(),
            initializer=tf.keras.initializers.RandomUniform(seed=11),
            trainable=(kind == "trainable_quantum"),
            name="ablation_quantum",
        )(circuits)
        inputs = [features, circuits]
    elif kind == "matched_classical":
        branch = dense_layer(2, activation="tanh", seed=11)(features)
    elif kind != "classical_only":
        raise ValueError("Unknown ablation variant")
    if kind != "classical_only":
        shared = tf.keras.layers.Concatenate()([shared, branch])
    output = dense_layer(1, activation="sigmoid")(shared)
    candidate = tf.keras.Model(inputs, output)
    candidate.compile(
        optimizer=tf.keras.optimizers.Adam(0.02),
        loss="binary_crossentropy", metrics=["accuracy"],
    )
    return candidate, uses_quantum

ablation_models, ablation_records = {}, []
for kind in ("classical_only", "fixed_quantum",
             "trainable_quantum", "matched_classical"):
    candidate, uses_quantum = build_ablation(kind)
    train_inputs = ([x_train_scaled, train_circuits]
                    if uses_quantum else x_train_scaled)
    valid_inputs = ([x_valid_scaled, valid_circuits]
                    if uses_quantum else x_valid_scaled)
    started = time.perf_counter()
    run = candidate.fit(
        train_inputs, y_train, batch_size=16, epochs=20, verbose=0,
        validation_data=(valid_inputs, y_valid),
    )
    ablation_records.append({
        "variant": kind,
        "trainable_parameters": sum(
            int(np.prod(v.shape)) for v in candidate.trainable_variables
        ),
        "final_val_accuracy": run.history["val_accuracy"][-1],
        "training_seconds": time.perf_counter() - started,
    })
    ablation_models[kind] = candidate
for record in ablation_records:
    print(record)


In [ ]:
optimizer = tf.keras.optimizers.Adam(0.01)
loss_fn = tf.keras.losses.BinaryCrossentropy()

@tf.function
def train_step(inputs, labels):
    """Apply one controlled-model update and return loss and gradient norms."""
    with tf.GradientTape() as tape:
        predictions = hybrid_model(inputs, training=True)
        loss = loss_fn(labels, predictions)
    gradients = tape.gradient(loss, hybrid_model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, hybrid_model.trainable_variables))
    gradient_report = {
        variable.name: tf.linalg.global_norm([gradient])
        for variable, gradient in zip(
            hybrid_model.trainable_variables, gradients
        ) if gradient is not None
    }
    return loss, gradient_report

batch_inputs = [x_train_scaled[:8], train_circuits[:8]]
batch_loss, gradient_report = train_step(batch_inputs, y_train[:8])
print("batch loss:", float(batch_loss))
for name, norm in gradient_report.items():
    print(name, float(norm))


In [ ]:
class ReUploading(tf.keras.layers.Layer):
    """Re-encode two numeric features between trainable circuit blocks."""

    def __init__(self, depth):
        super().__init__()
        if depth < 1:
            raise ValueError("depth must be positive")
        self.depth = depth
        self.weight_symbols = sympy.symbols(f"w_0:{2 * depth}")
        self.data_symbols = sympy.symbols(f"x_0:{2 * depth}")
        self.circuit = cirq.Circuit()
        for layer in range(depth):
            for j, qubit in enumerate(qubits):
                index = 2 * layer + j
                self.circuit += cirq.ry(
                    self.data_symbols[index]
                )(qubit)
                self.circuit += cirq.rx(
                    self.weight_symbols[index]
                )(qubit)
            self.circuit += cirq.CZ(*qubits)
        names = [
            str(s) for s in self.weight_symbols + self.data_symbols
        ]
        self.order = [names.index(name) for name in sorted(names)]
        self.angles = self.add_weight(
            name="quantum_angles", shape=(1, 2 * depth),
            initializer=tf.keras.initializers.RandomUniform(seed=7),
        )
        self.empty = tfq.convert_to_tensor([cirq.Circuit()])
        self.executor = tfq.layers.ControlledPQC(
            self.circuit, observables,
            backend="noiseless", repetitions=None,
            differentiator=tfq.differentiators.ParameterShift(),
        )

    def call(self, inputs):
        """Return two expectations per row of a (batch, 2) angle tensor."""
        batch = tf.shape(inputs)[0]
        weights = tf.tile(self.angles, [batch, 1])
        data = tf.tile(inputs, [1, self.depth])
        values = tf.gather(
            tf.concat([weights, data], axis=1), self.order, axis=1
        )
        circuits = tf.repeat(self.empty, repeats=batch)
        return self.executor([circuits, values])

reupload_probe = ReUploading(depth=2)
probe_features = tf.Variable(x_train_scaled[:2])
with tf.GradientTape() as tape:
    reupload_outputs = reupload_probe(probe_features)
    reupload_sum = tf.reduce_sum(reupload_outputs)
input_gradient = tape.gradient(reupload_sum, probe_features)
print("re-uploaded output shape:", reupload_outputs.shape)
print("input gradient norm:", float(tf.linalg.global_norm([input_gradient])))


In [ ]:
import pandas as pd
import time

records = []
for depth in [1, 2, 3, 4]:
    tf.keras.utils.set_random_seed(7)
    block = ReUploading(depth)
    depth_model = tf.keras.Sequential([
        tf.keras.Input(shape=(2,)), block,
        dense_layer(1, activation="sigmoid"),
    ])
    depth_model.compile(
        optimizer=tf.keras.optimizers.Adam(0.02),
        loss="binary_crossentropy", metrics=["accuracy"],
    )
    started = time.perf_counter()
    run = depth_model.fit(
        x_train_scaled, y_train, batch_size=16, epochs=20, verbose=0,
        validation_data=(x_valid_scaled, y_valid),
    )
    elapsed = time.perf_counter() - started
    with tf.GradientTape() as tape:
        loss = tf.keras.losses.binary_crossentropy(
            y_train, depth_model(x_train_scaled)
        )
        loss = tf.reduce_mean(loss)
    gradients = tape.gradient(loss, block.trainable_variables)
    records.append({
        "depth": depth,
        "quantum_parameters": int(np.prod(block.angles.shape)),
        "operations": len(list(block.circuit.all_operations())),
        "final_val_accuracy": run.history["val_accuracy"][-1],
        "quantum_grad_norm": float(tf.linalg.global_norm(gradients)),
        "seconds_per_epoch": elapsed / 20,
    })
print(pd.DataFrame(records).to_string(index=False))
